##import libraries

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os

##load data

In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# چون MNIST سیاه‌سفیده، باید به ۳ کانال تبدیل کنیم
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.Grayscale(num_output_channels=3),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

train_dataset = datasets.MNIST(root='./data', train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root='./data', train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)


100%|██████████| 9.91M/9.91M [00:00<00:00, 18.1MB/s]
100%|██████████| 28.9k/28.9k [00:00<00:00, 472kB/s]
100%|██████████| 1.65M/1.65M [00:00<00:00, 4.43MB/s]
100%|██████████| 4.54k/4.54k [00:00<00:00, 12.4MB/s]


##load model

In [ ]:
model = models.mobilenet_v3_large(pretrained=True)

for param in model.features.parameters():
    param.requires_grad = False

num_features = model.classifier[3].in_features
model.classifier[3] = nn.Linear(num_features, 10)

model = model.to(device)


/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Large_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Large_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Downloading: "https://download.pytorch.org/models/mobilenet_v3_large-8738ca79.pth" to /root/.cache/torch/hub/checkpoints/mobilenet_v3_large-8738ca79.pth


100%|██████████| 21.1M/21.1M [00:01<00:00, 18.0MB/s]


##difine optimizer

In [4]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=0.001)

##training model

In [6]:
save_path = '/content/drive/MyDrive/DATA/model for mnist'
os.makedirs(save_path, exist_ok=True)

num_epochs = 45

for epoch in range(num_epochs):
    model.train()
    running_loss = 0.0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    avg_loss = running_loss / len(train_loader)
    print(f"Epoch [{epoch+1}/{num_epochs}], Loss: {avg_loss:.4f}")

    if (epoch + 1) % 5 == 0:
        checkpoint_name = f"mobilenetv3_large_mnist_epoch_{epoch+1}.pth"
        torch.save({
            'epoch': epoch + 1,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'loss': avg_loss
        }, os.path.join(save_path, checkpoint_name))
        print(f"✅ Model saved at epoch {epoch+1}")


Epoch [1/45], Loss: 0.2141
Epoch [2/45], Loss: 0.1103
Epoch [3/45], Loss: 0.0891
Epoch [4/45], Loss: 0.0738
Epoch [5/45], Loss: 0.0635
✅ Model saved at epoch 5
Epoch [6/45], Loss: 0.0585
Epoch [7/45], Loss: 0.0524
Epoch [8/45], Loss: 0.0481
Epoch [9/45], Loss: 0.0435
Epoch [10/45], Loss: 0.0410
✅ Model saved at epoch 10
Epoch [11/45], Loss: 0.0388
Epoch [12/45], Loss: 0.0336
Epoch [13/45], Loss: 0.0320
Epoch [14/45], Loss: 0.0312
Epoch [15/45], Loss: 0.0343
✅ Model saved at epoch 15
Epoch [16/45], Loss: 0.0312
Epoch [17/45], Loss: 0.0284
Epoch [18/45], Loss: 0.0272
Epoch [19/45], Loss: 0.0256
Epoch [20/45], Loss: 0.0283
✅ Model saved at epoch 20
Epoch [21/45], Loss: 0.0274
Epoch [22/45], Loss: 0.0233
Epoch [23/45], Loss: 0.0246
Epoch [24/45], Loss: 0.0215
Epoch [25/45], Loss: 0.0249
✅ Model saved at epoch 25
Epoch [26/45], Loss: 0.0208
Epoch [27/45], Loss: 0.0212
Epoch [28/45], Loss: 0.0218
Epoch [29/45], Loss: 0.0231
Epoch [30/45], Loss: 0.0194
✅ Model saved at epoch 30
Epoch [31/45],

##model evaluation

In [7]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1)
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"🎯 Accuracy on MNIST test set: {accuracy:.2f}%")

🎯 Accuracy on MNIST test set: 97.90%
